# Diffusion-Limited Heat Treatment in a Slab

This notebook models one-dimensional thermal diffusion in a slab with fixed-temperature boundaries. The heat equation is

$$\partial_t u = -\alpha L u,$$

where $L$ is the positive finite-difference Dirichlet Laplacian. A bounded exponential polynomial approximates the heat propagator $\exp(-\alpha t L)$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from qsvt.matrix_functions import design_imaginary_time_polynomial
from qsvt.pde import dirichlet_laplacian_1d
from qsvt.polynomials import eval_polynomial
from qsvt.rescaling import rescale_positive_semidefinite
from qsvt.spectral import (
    apply_function_to_hermitian,
    apply_polynomial_to_hermitian,
    eigh_hermitian,
)

np.set_printoptions(precision=4, suppress=True)

In [ ]:
n_points = 48
length = 1.0
alpha = 0.002
time = 0.08
x, laplacian = dirichlet_laplacian_1d(n_points, length=length)

initial_temperature = np.exp(-0.5 * ((x - 0.35) / 0.07) ** 2)
initial_temperature += 0.6 * np.exp(-0.5 * ((x - 0.68) / 0.05) ** 2)

scaled = rescale_positive_semidefinite(laplacian)
poly = design_imaginary_time_polynomial(alpha * time, scaled.scale, degree=24)
polynomial_heat_operator = poly.prefactor * apply_polynomial_to_hermitian(
    scaled.matrix, poly.coeffs
)

exact_heat_operator = apply_function_to_hermitian(
    laplacian, lambda lam: np.exp(-alpha * time * lam)
)

exact_temperature = exact_heat_operator @ initial_temperature
poly_temperature = polynomial_heat_operator @ initial_temperature
relative_error = np.linalg.norm(poly_temperature - exact_temperature) / np.linalg.norm(
    exact_temperature
)
relative_error

In [ ]:
evals, _ = eigh_hermitian(laplacian)
scaled_lam = evals / scaled.scale
exact_decay = np.exp(-alpha * time * evals)
poly_decay = poly.prefactor * eval_polynomial(poly.coeffs, scaled_lam)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), constrained_layout=True)

axes[0].plot(x, initial_temperature, label="initial")
axes[0].plot(x, exact_temperature, label="exact heat flow")
axes[0].plot(x, poly_temperature, "--", label="polynomial heat flow")
axes[0].set_xlabel("position in slab")
axes[0].set_ylabel("temperature")
axes[0].legend()

axes[1].plot(evals, exact_decay, label="exact")
axes[1].plot(evals, poly_decay, "--", label="polynomial")
axes[1].set_xlabel("Laplacian eigenvalue")
axes[1].set_ylabel("thermal damping")
axes[1].legend()

fig.suptitle("Polynomial approximation to heat diffusion")
plt.show()

In [ ]:
assert np.linalg.norm(exact_temperature) < np.linalg.norm(initial_temperature)
assert relative_error < 0.03
assert np.max(np.abs(poly_decay - exact_decay)) < 0.03

print(f"relative_temperature_error: {relative_error:.4f}")
print(f"initial_norm: {np.linalg.norm(initial_temperature):.4f}")
print(f"cooled_norm: {np.linalg.norm(exact_temperature):.4f}")
print("validation: passed")